In [1]:
import torch 
import torch.nn as nn

In [2]:
class NoisyTopKGating(nn.Module):
    def __init__(self, D, N, K):
        super(NoisyTopKGating, self).__init__()
        self.W_G = torch.nn.Parameter(torch.randn((D, N)))
        self.W_N = torch.nn.Parameter(torch.randn((D, N)))
        self.normal_dist = torch.distributions.Normal(loc=0, scale=1)
        self.softplus = nn.Softplus()

        self.D = D
        self.N = N 
        self.K = K


    def forward(self, X:torch.tensor): # (B, S, D)
        (B, S, D) = X.shape
        W_G = X @ self.W_G # (B, S, D) @ (D, N) = (B, S, N)
        W_N = X @ self.W_N

        assert W_G.shape == (B, S, self.N)
        
        e = self.normal_dist.sample((B, S, self.N)) 

        H = W_G + e * self.softplus(W_N) # (B, S, N)
        KV, KI = torch.topk(H, k=self.K, dim=-1) # (B, S, K)

        mask = torch.ones_like(H, dtype=torch.bool)
        mask.scatter_(dim=-1, index=KI, value=False)

        H = H.masked_fill(mask, float("-inf"))


        assert KI.shape == (B, S, self.K)


        G = nn.Softmax(dim=-1)(H)

        assert G.shape == (B, S, self.N)

        probs = G.mean(dim=(0, 1))

        assert probs.shape == (self.N, )

        threshold_logit = KV[:,:,-1:]

        assert threshold_logit.shape == (B, S, 1)

        D = (W_G - threshold_logit) / W_N 

        assert D.shape == (B, S, self.N)

        f = self.normal_dist.cdf(D).mean(dim=(0, 1))

        assert f.shape == (self.N, )

        aux_loss = torch.sum(f * probs)

        return G, aux_loss, KI     



        



noisy_top_k_gating = NoisyTopKGating(D=2, N=5, K=3)

X = torch.randn((1, 4, 2)).float()
noisy_top_k_gating.forward(X)

(tensor([[[0.2645, 0.0000, 0.3774, 0.0000, 0.3581],
          [0.2401, 0.0000, 0.2918, 0.0000, 0.4681],
          [0.0000, 0.1972, 0.3935, 0.4094, 0.0000],
          [0.2994, 0.2943, 0.0000, 0.0000, 0.4063]]],
        grad_fn=<SoftmaxBackward0>),
 tensor(0.4976, grad_fn=<SumBackward0>),
 tensor([[[2, 4, 0],
          [4, 2, 0],
          [3, 2, 1],
          [4, 0, 1]]]))

In [3]:
a = torch.randn((1, 4, 3))
print(a)

b = torch.tensor([
    [1, 2],
    [2, 0],
    [0, 1],
    [1, 0]
    ]).unsqueeze(0)

print(b.shape)
torch.gather(a, dim=-1, index=b)

tensor([[[-0.2579,  0.1810, -0.7184],
         [-1.2165,  0.6958,  0.4915],
         [ 0.4549, -0.9211,  0.9594],
         [ 1.9944,  0.2996, -0.2642]]])
torch.Size([1, 4, 2])


tensor([[[ 0.1810, -0.7184],
         [ 0.4915, -1.2165],
         [ 0.4549, -0.9211],
         [ 0.2996,  1.9944]]])

In [ ]:
class ShazeerMOE(nn.Module):
    def __init__(self, D, N, K):
        super(ShazeerMOE, self).__init__()
        self.D = D
        self.N = N 
        self.K = K
        self.block = nn.ModuleList([nn.Linear(D, D), nn.GELU(), nn.Linear(D, D)])
        self.experts = []
        for _ in range(N):
            new_block = self.block
            self.experts.append(new_block)
        self.experts = nn.ModuleList(self.experts)

        self.noisy_gating = NoisyTopKGating(D, N, K)

    def forward(self, X):
        B, S, _ = X.shape
        G, aux_loss, KI = self.noisy_gating(X)

        XE = torch.zeros((N, B, S, D))
        for i in range(len(self.experts)):
            XE[i] = self.experts[i](X)
        
        XE = XE.permute((1, 2, 3, 0)) # (B, S, D, N)
        XEK = torch.gather(self.experts, dim=-1, index=KI) # (B, S, D, K)
        
        G_routed = torch.gather(G, dim=-1, index=KI) #(B, S, K)
        
        y =  torch.einsum("bsdk,bsk->bsd", XEK, G_routed)

        return y, aux_loss

D = 2
N = 4
K = 2
B = 1
S = 3
shazeer_moe = ShazeerMOE(D, N, K)
a = torch.randn((B, S, D))
shazeer_moe.forward(a)


NotImplementedError: Module [ModuleList] is missing the required "forward" function